# Phase 1 bootstrap — VM thue (Ubuntu 24.04 + RTX 3090 Ti 24GB)

**Notebook nay chay tuyen tinh: `Run All` = chay tron bo pipeline train lai tu dau.**
Cac buoc doi chieu / tuy chon da duoc don xuong **PHU LUC o cuoi file** de khong bi
kich hoat nham khi Run All.

Truoc khi chay: dua `phase1_project.zip` len **cung thu muc** voi notebook nay
(`scp phase1_project.zip user@host:~/` hoac keo-tha qua file browser cua Jupyter).

Thu tu bat buoc (moi buoc doc output cua buoc truoc):
`generate_attempts` (GPU/vLLM) -> `build_dataset` (API, khong GPU) ->
`train_sft` (GPU/Unsloth) -> `evaluate_self_correction` (GPU/Unsloth).

Moi buoc deu `tee` ra `AGI/outputs/logs/*.log` de doc lai khi debug.

## 0. Preflight — kiem tra GPU TRUOC KHI lam bat cu thu gi

Dung lai ngay neu may khong dat chuan, thay vi phat hien sau khi da cai 10 phut
dependencies. Nguong cung: **compute capability >= 7.5**. Duoi nguong do Unsloth
va vLLM deu khong chay (P40/P100 = 6.1, V100 = 7.0 -> fail).

In [ ]:
import os
import shutil
import subprocess

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

raw = sh("nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv,noheader")
if not raw:
    raise RuntimeError("nvidia-smi khong chay duoc -- khong thay GPU. Dung lai.")

name, cc_str, vram = [x.strip() for x in raw.split("\n")[0].split(",")]
cc = float(cc_str)

try:
    ram = [l for l in sh("free -g").split("\n") if l.startswith("Mem:")][0].split()[1] + " GB"
except Exception:
    ram = "n/a"

print(f"GPU        : {name}")
print(f"Compute cap: {cc}")
print(f"VRAM       : {vram}")
print(f"RAM        : {ram}")
print(f"Disk free  : {shutil.disk_usage('.').free / 1e9:.0f} GB")
print(f"Python     : {sh('python3 --version')}")

if cc < 7.5:
    raise RuntimeError(
        f"GPU nay CC={cc}, duoi nguong 7.5. Unsloth va vLLM deu KHONG chay duoc "
        f"(Pascal 6.1 / Volta 7.0). Doi may khac -- xem README muc 'Choosing a rental GPU'."
    )
if cc < 8.0:
    print(f"\n[CANH BAO] CC={cc} (Turing). Chay duoc nhung khong co bf16 va khong co "
          f"FlashAttention-2 -- se cham hon dang ke. Nen dung Ampere tro len.")
else:
    print(f"\n[OK] CC={cc} >= 8.0: co bf16 + FlashAttention-2 + tensor cores.")

# Duong dan 2 venv rieng (xem Buoc 3) -- dung lai o tat ca cac cell sau.
HOME = os.path.expanduser("~")
VLLM_PY = f"{HOME}/venv-vllm/bin/python"
UNSLOTH_PY = f"{HOME}/venv-unsloth/bin/python"
SUDO = "" if os.geteuid() == 0 else "sudo "
print(f"\nvLLM python    : {VLLM_PY}")
print(f"Unsloth python : {UNSLOTH_PY}")

## 1. Giai nen project

In [ ]:
!{SUDO}apt-get install -y -qq unzip > /dev/null 2>&1 || true
!unzip -o -q phase1_project.zip -d AGI
!mkdir -p AGI/outputs/logs AGI/data/processed
!find AGI -maxdepth 2 -type d | sort

## 2. Tao `.env`

Dien key that vao 2 bien ngay ben duoi. Cell nay **raise neu key con trong** — co y
de `Run All` dung lai ngay tai day, thay vi ghi ra mot `.env` hong roi fail o tan
buoc `build_dataset` (sau khi da ton GPU cho `generate_attempts`).

In [ ]:
from pathlib import Path

DEEPSEEK_API_KEY = ""   # <-- DIEN KEY THAT VAO DAY
HF_TOKEN         = ""   # <-- token HF (write): can cho Buoc 8 push va PHU LUC A (repo private)

if not DEEPSEEK_API_KEY:
    raise ValueError("Chua dien DEEPSEEK_API_KEY. Dien vao roi chay lai cell nay.")
if not HF_TOKEN:
    print("[CANH BAO] HF_TOKEN trong -- Buoc 8 (push) va PHU LUC A se fail. "
          "Cac buoc train/eval local van chay binh thuong.")

Path("AGI/.env").write_text(
    f"DEEPSEEK_MODEL=op/z-ai/glm-5.2\n"
    f"DEEPSEEK_API_KEY={DEEPSEEK_API_KEY}\n"
    f"DEEPSEEK_BASE_URL=https://api.vilao.ai/v1\n"
    f"HF_TOKEN={HF_TOKEN}\n",
    encoding="utf-8",
)
print("Da ghi AGI/.env")

## 3. Cai dependencies — **2 venv rieng biet, qua `uv`**

vLLM va Unsloth moi ben keo mot bo `torch`/CUDA rieng; cai chung 1 moi truong thi ben
nay de ghi de ben kia, trieu chung la loi import la o giua chung pipeline. Ban Colab cu
xu ly bang cach "Restart session + cai lai unsloth" o giua — tren VM thue thi khong can
chap nhan su mong manh do nua.

- `~/venv-vllm`    -> `generate_attempts.py` + `evaluate_self_correction_vllm.py`
- `~/venv-unsloth` -> `build_dataset.py` + `train_sft.py` (+ ban eval Unsloth du phong)

**Vi sao `uv` chu khong phai `apt-get` + `pip`:** da gap thuc te tren VM thue -- may
KHONG co `pip`, KHONG co `python3-venv`, va `sudo` doi password. Ke hoach
`apt-get install python3-venv python3-pip` chet ngay tu dau. `uv` la binary tinh cai
vao `~/.local/bin`, tu tao venv (khong di qua `ensurepip` nen khong can `python3-venv`)
va tu cai package (khong can `pip`) -- ca 2 thu dang thieu deu tro nen khong lien quan.
Kem theo: nhanh hon nhieu va co cache chung giua 2 venv (phan `torch`/`nvidia-*` trung
nhau chi tai 1 lan). Ubuntu 24.04 con co PEP 668 chan pip cai thang vao python he
thong -- dung venv la cach xu ly dung cho viec do.

Cell nay mat ~10-20 phut, chay 1 lan moi khi thue may moi.

In [ ]:
# uv: khong can sudo, cai vao ~/.local/bin
!curl -LsSf https://astral.sh/uv/install.sh | sh
UV = f"{HOME}/.local/bin/uv"
!{UV} --version

In [ ]:
# --- venv 1: vLLM (generate_attempts + eval) ---
# KHONG cai requirements.txt o day: file do co 'unsloth', se keo torch cua Unsloth vao
# dung cai venv dang co gang giu sach cho vLLM.
# datasets + huggingface_hub la cho src/eval_common.py (tai GSM8K/MBPP, tai adapter),
# sympy cho MathVerifier.
!{UV} venv {HOME}/venv-vllm --python 3.12
!{UV} pip install --python {VLLM_PY} vllm python-dotenv pyyaml sympy datasets huggingface_hub

print("=== venv-vllm ===")
!{VLLM_PY} -c "import torch, vllm, datasets; print('torch', torch.__version__, '| vllm', vllm.__version__, '| cuda', torch.cuda.is_available())"

In [ ]:
# --- venv 2: Unsloth (train_sft + build_dataset) ---
!{UV} venv {HOME}/venv-unsloth --python 3.12
!{UV} pip install --python {UNSLOTH_PY} unsloth
!{UV} pip install --python {UNSLOTH_PY} -r AGI/requirements.txt

print("=== venv-unsloth ===")
!{UNSLOTH_PY} -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available()); x = torch.randn(4,4,device='cuda') @ torch.randn(4,4,device='cuda'); torch.cuda.synchronize(); print('real compute ok', tuple(x.shape))"

### 3b. Helper doc log eval (ghi 1 lan, dung lai o Buoc 7 va PHU LUC A)

Tach `thinking` / `phat_hien_loi` / `nguyen_nhan` / `sua_lai` ra khoi `correction_text`,
kem `attempt_text` + `verifier_detail` de doi chieu xem model co that su "biet sai" hay
chi doan mo / chep dap so. In them breakdown tach rieng `format_incomplete` (bi cat
ngang, chua toi `### Sua lai`) khoi `genuine_still_wrong` — hai cai nay ban chat khac
nhau, gop lai se doc sai ket qua.

In [ ]:
from pathlib import Path

SCRIPT = r'''
# Doc <log>.jsonl cua evaluate_self_correction.py -> ghi <log>_readable.jsonl + in breakdown.
# Dung: python report_eval_log.py outputs/eval_new_adapter.jsonl
import json
import re
import sys
from collections import defaultdict
from pathlib import Path

IN_PATH = sys.argv[1] if len(sys.argv) > 1 else "outputs/eval_self_correction_log.jsonl"

_THINKING_RE = re.compile(r"<thinking>\s*(.*?)\s*</thinking>", re.DOTALL)
_SECTION_RE = re.compile(
    r"###\s*Ph[aá]t hi[eệ]n l[oỗ]i\s*\n(.*?)\n\n###\s*Nguy[eê]n nh[aâ]n\s*\n(.*?)\n\n###\s*S[uử]a l[aạ]i\s*\n(.*)",
    re.DOTALL,
)

records = []
with open(IN_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

out_path = str(Path(IN_PATH).with_name(Path(IN_PATH).stem + "_readable.jsonl"))
with open(out_path, "w", encoding="utf-8") as f:
    for r in records:
        enriched = dict(r)
        ct = r.get("correction_text", "")
        tm = _THINKING_RE.search(ct)
        sm = _SECTION_RE.search(ct)
        enriched["thinking"] = tm.group(1).strip() if tm else None
        enriched["phat_hien_loi"] = sm.group(1).strip() if sm else None
        enriched["nguyen_nhan"] = sm.group(2).strip() if sm else None
        enriched["sua_lai"] = sm.group(3).strip() if sm else None
        f.write(json.dumps(enriched, ensure_ascii=False) + "\n")

roles = {r.get("reflect_role") for r in records if r.get("reflect_role")}
print(f"Doc tu      : {IN_PATH}")
print(f"Ghi ra      : {out_path}  ({len(records)} dong)")
print(f"reflect_role: {roles or '(khong ro -- log ban cu)'}")

domains = ["math", "code"]
stats = {d: defaultdict(int) for d in domains}
stats["total"] = defaultdict(int)
for r in records:
    for key in (r["domain"], "total"):
        s = stats[key]
        s["n"] += 1
        if r.get("initial_passed"):
            s["initial_correct"] += 1
        else:
            s["initial_wrong"] += 1
            if r.get("format_incomplete"):
                s["format_incomplete"] += 1
            elif r.get("second_passed"):
                s["self_corrected"] += 1
            else:
                s["genuine_still_wrong"] += 1

hdr = f"{'domain':<8}{'n':>4}{'init_ok':>9}{'init_wrong':>12}{'fmt_incompl':>13}{'self_corr':>11}{'still_wrong':>13}"
print("\n" + hdr)
print("-" * len(hdr))
for key in domains + ["total"]:
    s = stats[key]
    print(f"{key:<8}{s['n']:>4}{s['initial_correct']:>9}{s['initial_wrong']:>12}"
          f"{s['format_incomplete']:>13}{s['self_corrected']:>11}{s['genuine_still_wrong']:>13}")

print()
for key in domains + ["total"]:
    s = stats[key]
    if s["initial_wrong"]:
        print(f"Ty le tu sua THAT [{key:<5}]: {s['self_corrected'] / s['initial_wrong'] * 100:5.1f}%"
              f"   ({s['self_corrected']}/{s['initial_wrong']})")
'''

Path("AGI/report_eval_log.py").write_text(SCRIPT, encoding="utf-8")
print("Da ghi AGI/report_eval_log.py")

## 4. Sinh attempt cho problem moi — vLLM (GPU)

`data/problems/*.jsonl` da mo rong len 1300 bai (776 toan + 524 code) lay tu split
`train` (GSM8K) / `full/train` (MBPP) — **tach biet hoan toan** voi split `test` ma
held-out eval dung, nen khong lam ho eval.

**Resumable:** script tu bo qua problem_id da co attempt, chi sinh cho bai moi. An toan
khi chay lai sau khi bi ngat.

vLLM chay bf16 thang, **khong dung bitsandbytes** (`generation.load_in_4bit: false`
trong `configs/phase1.yaml`) — 7B bf16 ~15GB, vua trong 24GB, con ~5GB KV cache. Day
cung la cach tranh lop loi CUDA runtime da lam fail lan chay tren T4.

In [ ]:
!cd AGI && {VLLM_PY} -m src.generate_attempts 2>&1 | tee outputs/logs/generate_attempts.log

In [ ]:
!wc -l AGI/data/processed/attempts.jsonl

## 5. Build dataset — verify -> critique (API) -> re-verify -> append (KHONG can GPU)

**Resumable:** tu bo qua `problem_id` da xu ly (ghi trong `build_dataset_seen_ids.txt`),
chi ton API call cho bai moi.

Chu y phan stats in ra o cuoi: `total / already_correct / kept / discarded`. Keep rate
lich su khoang 40% — `already_correct` (model giai dung ngay, khong co gi de sua) va
correction cua model lon truot re-verify deu bi loai. **Loai la co chu dich**, khong
phai bug can toi uu cho keep rate cao hon.

In [ ]:
!cd AGI && {UNSLOTH_PY} -m src.build_dataset 2>&1 | tee outputs/logs/build_dataset.log

In [ ]:
!wc -l AGI/data/processed/phase1_sft.jsonl

## 6. Train QLoRA SFT (GPU / Unsloth)

Giu `small_model.load_in_4bit: true` (QLoRA) du may du VRAM — con so 40.5% hien tai do
tren adapter QLoRA, doi sang LoRA bf16 se lam ket qua moi khong con so sanh duoc.
`training.max_seq_length` da nang lai 4096 (tung phai ha 2048 vi card 16GB).

**Neu OOM:** ha `max_seq_length` ve 2048 trong `configs/phase1.yaml` truoc, dung dong
vao batch size.

In [ ]:
!cd AGI && {UNSLOTH_PY} -m src.train_sft 2>&1 | tee outputs/logs/train_sft.log

## 7. Held-out eval — day moi la con so co y nghia

Training loss KHONG cho biet model co hoc duoc tu sua hay khong. Metric that la
`self_corrected / initial_wrong`.

Held-out = offset 150 tro di, tach roi hoan toan voi phan da dung de train.
`--reflect-role` mac dinh da la `tool` (dung mac dinh, dung tu them vao).

**Chay bang vLLM, khong phai Unsloth.** Phep do nay von co cau truc 2 pha (tat ca bai
tu giai -> verify -> chi bai sai moi phan tu), ca 2 pha deu la batch thuan -> vLLM chi
can 2 lan `generate()` thay vi ~2 lan/bai tuan tu.

**100 toan + 100 code, khong phai 30+30.** Mau so cua metric la `initial_wrong`; o
60 bai thi mau so chi ~35, chenh 1 case lam ty le nhay ~3% -- lon hon hieu ung dang do.
Nho batching cua vLLM, 200 bai gan nhu khong dat hon 60 bai (chi phi chinh la load
model). Con du de: GSM8K test con 1169 bai sau offset 150, MBPP test con 350.

**Luu y khi doi chieu:** doi engine thi kernel + RNG khac, nen khong bao gio trung
tuyet doi voi so cu do bang Unsloth. Voi `temperature=0.7` thi 2 lan chay cung engine
cung da khac nhau. Muon chac chan hon thi tang so bai, dung ky vong 2 engine ra cung
mot con so.

In [ ]:
!cd AGI && {VLLM_PY} -m src.evaluate_self_correction_vllm \
    --math-offset 150 --math-limit 100 --code-offset 150 --code-limit 100 \
    --log-file outputs/eval_new_adapter.jsonl 2>&1 | tee outputs/logs/eval_new_adapter.log

In [ ]:
!cd AGI && {VLLM_PY} report_eval_log.py outputs/eval_new_adapter.jsonl

**Moc so sanh** (xem README):

| Run | Math | Code | Overall |
|---|---|---|---|
| Baseline (role `user`) | 0.0% | 28.6% | 22.9% |
| Adapter cu + role `tool` khi eval | 33.3% | 28.6% | 29.7% |
| Retrain role `tool` (135 mau) | 22.2% | 46.4% | **40.5%** |
| Lan nay (~1300 bai) | ? | ? | ? |

Muc tieu ke hoach: >= 70%. Nut that tu truoc den gio la **so luong sample** (135 vs
800-1500), nen lan chay nay la phep thu truc tiep cho gia thuyet do.

## 8. (Tuy chon) Push adapter len HF Hub

Ghi de `Kxck/AGI_v1` (van private). Can `HF_TOKEN` co quyen write o Buoc 2.

In [ ]:
!cd AGI && {UNSLOTH_PY} -m src.push_to_hub --local-dir outputs/phase1_lora

---
---

# PHU LUC — chay tay, KHONG nam trong luong Run All

Cac cell duoi day khong can cho pipeline chinh. Chung duoc don xuong day co y: truoc kia
chung nam giua notebook (Buoc 5 / 5b cu) va bi kich hoat nham moi lan Run All, lam ton
GPU va ghi de log.

## PHU LUC A. Eval adapter CU tren Hub (`Kxck/AGI_v1`) de doi chieu

Chi chay khi muon so sanh truc tiep adapter cu vs adapter vua train. Can `HF_TOKEN` that
(repo private).

Luu y: tinh den lan cap nhat README gan nhat, `Kxck/AGI_v1` **van dang giu ban role
`user` cu**, chua duoc push lai ban role `tool`. Neu Buoc 8 o tren da chay thi repo nay
da bi ghi de bang adapter moi, va cell nay khong con la "adapter cu" nua.

In [ ]:
!cd AGI && {VLLM_PY} -m src.evaluate_self_correction_vllm --adapter Kxck/AGI_v1 \
    --math-offset 150 --math-limit 100 --code-offset 150 --code-limit 100 \
    --log-file outputs/eval_old_hub_adapter.jsonl 2>&1 | tee outputs/logs/eval_old_hub_adapter.log

In [ ]:
!cd AGI && {VLLM_PY} report_eval_log.py outputs/eval_old_hub_adapter.jsonl

## PHU LUC B. Tien ich debug

Chay bat cu luc nao khi can xem tinh trang may hoac doc lai log cu.

In [ ]:
# VRAM dang dung + tien trinh GPU
!nvidia-smi

In [ ]:
# Kiem tra khong co 2 tien trinh cung ghi 1 file output (gay hong file -- da tung xay ra)
!ps aux | grep -E "generate_attempts|build_dataset" | grep -v grep || echo "(khong co tien trinh nao dang chay -- an toan)"

In [ ]:
# Xem 40 dong cuoi cua tung log
!for f in AGI/outputs/logs/*.log; do echo "=============== $f"; tail -n 40 "$f"; done

In [ ]:
# Kiem tra config THAT SU dang duoc dung (doc qua chinh Config object, khong doan)
!cd AGI && {UNSLOTH_PY} -c "from src.config import load_config; c = load_config(); print('model            :', c.small_model['name_or_path']); print('train 4bit       :', c.small_model['load_in_4bit']); print('vllm  4bit       :', c.generation.get('load_in_4bit')); print('train max_seq_len:', c.training['max_seq_length']); print('epochs           :', c.training['epochs']); print('batch/accum      :', c.training['per_device_batch_size'], '/', c.training['gradient_accumulation_steps'])"